# M03D: Prompt Evaluation

Stop eyeballing responses. Measure prompt quality like a developer.

**Topics:**
- Labeled test set for systematic evaluation
- Eval harness that measures accuracy
- Before/after comparison framework

---

## 🔧 Step 1: Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import openai

load_dotenv(dotenv_path=Path("..") / ".env")

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"


def ask_openai(prompt, model=MODEL):
    """Send a prompt to OpenAI and return the response."""
    try:
        response = client.responses.create(
            model=model,
            input=prompt
        )
        return response.output_text.strip()
        
    except openai.AuthenticationError:
        return "Error: Invalid API key. Check your .env file."
    except openai.RateLimitError:
        return "Error: Rate limit exceeded. Wait and try again."
    except openai.APIConnectionError:
        return "Error: Network issue. Check your internet."
    except openai.BadRequestError:
        return "Error: Bad request. Check model name."
    except Exception as e:
        return f"API Error: {str(e)}"


print("✅ Ready!")

---

## 📊 Part 1: Build a Labeled Dataset

We start with **sentiment classification**. Each example has a `text` and a `label` (positive, negative, or neutral).

The label is our **ground truth** — we know the correct answer, and we'll compare the model's output against it. The label is never sent to the model.

We keep the dataset small (10 examples) so you can run evaluations quickly. Production evaluations typically use 50+ examples.

In [ ]:
examples = [
    {
        "text": "I love this product. It works great!",
        "label": "positive",
    },
    {
        "text": "This is the worst experience ever.",
        "label": "negative",
    },
    {
        "text": "It was okay, nothing special at all.",
        "label": "neutral",
    },
    {
        "text": "Absolutely fantastic service and staff.",
        "label": "positive",
    },
    {
        "text": "I am very disappointed with this.",
        "label": "negative",
    },
    {
        "text": "It does the job. I do not mind it.",
        "label": "neutral",
    },
    {
        "text": "I am thrilled with the results!",
        "label": "positive",
    },
    {
        "text": "I will never buy from them again.",
        "label": "negative",
    },
    {
        "text": "It is fine, but I would not recommend.",
        "label": "negative",
    },
    {
        "text": "The product is okay, but the shipping was terrible.",
        "label": "negative",
    },
]


# --------------------------------------------------------------
print(f"✅ Loaded {len(examples)} labeled examples")

---

## 📝 Part 2: Create a Baseline Prompt

A simple prompt with instructions but no examples. We'll measure its accuracy, then improve it.

In [ ]:
BASE_PROMPT_TEMPLATE = """
You are a sentiment classifier.

Your task is to read the text and decide if the
sentiment is "positive", "negative", or "neutral".

Text: {text}

Return only one word:
positive, negative, or neutral.
"""

# --------------------------------------------------------------
print("✅ BASE_PROMPT_TEMPLATE ready")

---

## 🔧 Part 3: Helper Function

Classifies text using a prompt template and normalizes the output to a label. This wraps `ask_openai` for the specific task of mapping raw model output to one of three labels.

In [ ]:
def classify_sentiment(text, prompt_template):
    """Classify sentiment using the given prompt template."""
    prompt = prompt_template.format(text=text)
    response = ask_openai(prompt).lower()

    if "positive" in response:
        return "positive"
    if "negative" in response:
        return "negative"
    if "neutral" in response:
        return "neutral"

    return response


# --------------------------------------------------------------
print("✅ classify_sentiment() ready")

### 🔍 Sanity Check

Before running the full evaluation, let's verify the helper works on a single example.

In [ ]:
# Quick sanity check: single prediction
sample = examples[0]
print(f"Text:      {sample['text']}")
print(f"Actual:   {classify_sentiment(sample['text'], BASE_PROMPT_TEMPLATE)}")
print(f"Expected: {sample['label']}")

---

## 📈 Part 4: Build the Evaluation Harness & Run Baseline

Run all examples through the prompt and measure accuracy. This is our mini **eval harness**.

### 🧩 The Evaluation Harness Pattern



A small, reusable function that runs your prompt against a fixed test set and prints metrics every time you change the prompt.

In [ ]:
def evaluate_prompt(prompt_template, examples):
    """Evaluate a prompt template on labeled examples."""
    
    # Step 1: Get model's answer for each example
    results = []
    for example in examples:
        actual = classify_sentiment(example["text"], prompt_template)
        results.append({
            "text": example["text"],
            "expected": example["label"],
            "actual": actual
        })

    # Step 2: Calculate accuracy
    correct = sum(1 for r in results if r["expected"] == r["actual"])
    total = len(results)
    accuracy = correct / total if total else 0.0

    print(f"Accuracy: {correct}/{total} = {accuracy:.2%}")

    # Step 3: Show errors (if any)
    errors = [r for r in results if r["expected"] != r["actual"]]
    
    if errors:
        print("\nErrors:")
        for error in errors:
            print(f"  Text: {error['text']}")
            print(f"  Expected: {error['expected']}, Actual: {error['actual']}")
    else:
        print("No errors on this test set!")

    return results


# --------------------------------------------------------------
print("✅ evaluate_prompt() ready")

#### Run Baseline Evaluation

In [ ]:
# --------------------------------------------------------------
# Evaluate baseline prompt
# --------------------------------------------------------------
print("📊 BASELINE EVALUATION")
print("="*60)

baseline_results = evaluate_prompt(
    BASE_PROMPT_TEMPLATE,
    examples
)

print("="*60)

---

## ✨ Part 5: Improve the Prompt

A better prompt with few-shot examples and stricter output instructions.

In [ ]:
IMPROVED_PROMPT_TEMPLATE = """
You are a strict sentiment classifier.

You must respond with exactly one word:
"positive", "negative", or "neutral".

Here are some examples:

Text: "I absolutely love this!"
Label: positive

Text: "This is the worst thing ever."
Label: negative

Text: "It is fine, I do not mind it."
Label: neutral

Now classify the new text.

Text: {text}

Answer with one word only:
positive, negative, or neutral.
"""

# --------------------------------------------------------------
print("✅ IMPROVED_PROMPT_TEMPLATE ready")

#### Run Improved Evaluation

In [ ]:
# --------------------------------------------------------------
# Evaluate improved prompt
# --------------------------------------------------------------
print("✨ IMPROVED EVALUATION")
print("="*60)

improved_results = evaluate_prompt(
    IMPROVED_PROMPT_TEMPLATE,
    examples
)

print("="*60)

---

## 🔍 Part 6: Compare Results

Don't just say "it feels better." See which predictions changed and whether they improved.



In [ ]:
def compare_results(old_results, new_results):
    """Show which answers changed between two evaluation runs."""
    print("Changed answers:\n")
    
    changed = False
    for i, (old, new) in enumerate(zip(old_results, new_results)):
        if old["actual"] != new["actual"]:
            changed = True
            print(f"Example {i}:")
            print(f"  Text: {old['text']}")
            print(f"  Expected: {old['expected']}")
            print(f"  Old: {old['actual']} → New: {new['actual']}")
            print()
    
    if not changed:
        print("No answers changed.")
    
    # Summary
    old_correct = sum(1 for r in old_results if r["expected"] == r["actual"])
    new_correct = sum(1 for r in new_results if r["expected"] == r["actual"])
    total = len(old_results)
    
    print(f"\nOld: {old_correct}/{total} | New: {new_correct}/{total}")
    if new_correct > old_correct:
        print("✅ New prompt is better")
    elif new_correct < old_correct:
        print("⚠️ Old prompt was better (regression)")
    else:
        print("→ Same accuracy")

# --------------------------------------------------------------
print("✅ compare_results() ready")

#### Compare Baseline vs Improved

In [ ]:
# --------------------------------------------------------------
# Compare baseline vs improved
# --------------------------------------------------------------
print("🔍 COMPARISON: Baseline → Improved")
print("="*60)

compare_results(
    baseline_results,
    improved_results
)

print("="*60)

> ⚠️ **Note:** With small test sets, a 1-2 example difference may be noise, not real improvement. For high-confidence comparisons, use larger test sets.

---

### 💪 Your Turn: Create Your Own Prompt

Beat the improved prompt's accuracy with your own design.

Try changing:
- The wording of the instructions
- The number or style of few-shot examples
- How strictly you specify the allowed labels

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise: Create Your Own Prompt
# --------------------------------------------------------------
# Objective: Beat the improved prompt's accuracy with your own design.
#
# 1. Create your prompt template (MY_PROMPT_TEMPLATE)
# 2. Run: evaluate_prompt(MY_PROMPT_TEMPLATE, examples)
# 3. Run: compare_results(improved_results, my_results)

MY_PROMPT_TEMPLATE = """
Your prompt here...

Text: {text}

Label:
"""

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Labeled Test Sets:**
- Create examples with known correct answers
- Cover positive, negative, and edge cases
- Keep sets small for fast iteration, scale up for production

**Eval Harness:**
- Automate accuracy measurement
- Show errors for debugging
- Run consistently across prompt versions

**Before/After Comparison:**
- Track which answers changed
- Verify changes are improvements
- Catch regressions early

**The Mindset:**
- Treat prompts like code
- Test before deploying
- Measure, don't guess

**The Flow:** Label data → Baseline prompt → Measure → Improve → Compare → Deploy winner

---

### 📍 Next Step

You've completed **Module 3: Production Prompting**!

**Up next — Module 4: Advanced Prompting**

**M04A: Instructions & Conversation Chaining** — Use the instructions parameter for persistent behavior and build multi-turn conversations.

---

## 🔧 Troubleshooting

**Model returning unexpected labels?**
- Check prompt clearly specifies exact label words
- Look at raw output before normalization
- Add more examples for edge cases

**Low accuracy even with a good-looking prompt?**
- Double-check that test labels are correct
- Some examples may be genuinely ambiguous
- Review the raw model output to see what it's actually returning

**API errors during evaluation?**
- Make sure `OPENAI_API_KEY` is valid and loaded
- Check if you're hitting rate limits
- Add a small delay between calls if needed

---